# EdgeTRM Consolidated Evaluation Notebook

This notebook consolidates the evaluation pipelines for all three EdgeTRM model variants:
1. **ARC-Prize 2024 (Baseline)**
2. **Maze (Hard Version)**
3. **Sudoku (Extreme Version)**

Experiments are evaluated across **3 random seeds (42, 43, 44)** for scientific stability and compared side-by-side in Pandas tables.

## Section 1 — Setup and global imports

In [ ]:
import sys, time, copy, json, math, warnings, io, os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## Section 2 — Model Configuration & Checkpoint Loading

In [ ]:
from trm import TinyRecursiveReasoningModel_ACTV1, TinyRecursiveReasoningModel_ACTV1Carry, TinyRecursiveReasoningModel_ACTV1InnerCarry
import yaml

# Flat ARC 2024 config directly satisfying TinyRecursiveReasoningModel_ACTV1Config validation rules
arc_config = {
    "batch_size": 32,
    "seq_len": 900,
    "num_puzzle_identifiers": 50911,
    "vocab_size": 12,
    "H_cycles": 3,
    "L_cycles": 4,
    "H_layers": 0,
    "L_layers": 2,
    "hidden_size": 512,
    "expansion": 4,
    "num_heads": 8,
    "pos_encodings": "rope",
    "halt_max_steps": 16,
    "halt_exploration_prob": 0.1,
    "forward_dtype": "bfloat16",
    "mlp_t": False,
    "puzzle_emb_ndim": 512,
    "puzzle_emb_len": 16,
    "no_ACT_continue": True
}

# Flat Maze Hard config
maze_config = {
    "batch_size": 32,
    "seq_len": 900,
    "num_puzzle_identifiers": 1,
    "vocab_size": 6,
    "H_cycles": 3,
    "L_cycles": 4,
    "H_layers": 0,
    "L_layers": 2,
    "hidden_size": 512,
    "expansion": 4,
    "num_heads": 8,
    "pos_encodings": "rope",
    "halt_max_steps": 16,
    "halt_exploration_prob": 0.1,
    "forward_dtype": "bfloat16",
    "mlp_t": False,
    "puzzle_emb_ndim": 512,
    "puzzle_emb_len": 16,
    "no_ACT_continue": True
}

# Flat Sudoku Extreme config
sudoku_config = {
    "batch_size": 32,
    "seq_len": 81,
    "num_puzzle_identifiers": 1,
    "vocab_size": 11,
    "H_cycles": 3,
    "L_cycles": 6,
    "H_layers": 0,
    "L_layers": 2,
    "hidden_size": 512,
    "expansion": 4,
    "num_heads": 8,
    "pos_encodings": "none",
    "halt_max_steps": 16,
    "halt_exploration_prob": 0.1,
    "forward_dtype": "bfloat16",
    "mlp_t": True,
    "puzzle_emb_ndim": 512,
    "puzzle_emb_len": 16,
    "no_ACT_continue": True
}

def get_inner(m):
    m2 = m.module if hasattr(m, 'module') else m
    return m2._orig_mod if hasattr(m2, '_orig_mod') else m2

def load_trm_model(name, checkpoint_path, config_dict):
    print(f"Loading {name} model from {checkpoint_path}...")
    model = TinyRecursiveReasoningModel_ACTV1(config_dict=config_dict)
    if not os.path.exists(checkpoint_path):
        print(f"[WARNING] Checkpoint {checkpoint_path} not found. Skipping weight loading.")
        return model
        
    state_dict = torch.load(checkpoint_path, map_location='cpu')
    if 'model' in state_dict:
        state_dict = state_dict['model']
        
    unwanted_prefix = '_orig_mod.model.'
    clean_state_dict = {}
    for k, v in state_dict.items():
        if k.startswith(unwanted_prefix):
            clean_state_dict[k[len(unwanted_prefix):]] = v
        else:
            clean_state_dict[k] = v
            
    # Robust resizing of the puzzle embedding weights to preserve learned parameters
    puzzle_emb_name = "inner.puzzle_emb.weights"
    expected_shape = model.inner.puzzle_emb.weights.shape
    if puzzle_emb_name in clean_state_dict:
        puzzle_emb = clean_state_dict[puzzle_emb_name]
        if puzzle_emb.shape != expected_shape:
            print(f"  Resizing puzzle embedding. Found {puzzle_emb.shape}, Expected {expected_shape}")
            new_weights = torch.empty(expected_shape, dtype=puzzle_emb.dtype, device=puzzle_emb.device)
            mean_emb = torch.mean(puzzle_emb, dim=0)
            new_weights[:] = mean_emb
            min_rows = min(puzzle_emb.shape[0], expected_shape[0])
            new_weights[:min_rows] = puzzle_emb[:min_rows]
            clean_state_dict[puzzle_emb_name] = new_weights
            
    model.load_state_dict(clean_state_dict, strict=False)
    model.__dict__['model'] = model
    model.eval()
    print(f"✓ {name} successfully loaded!")
    return model

In [ ]:
# Load checkpoints
arc_ckpt = "eval_checkpoint/step_10620"
maze_ckpt = "trm_maze_hard/model.pt"
sudoku_ckpt = "trm_sudoku_extreme/step_39060_sudoku_epoch_60k"

model_arc = load_trm_model("ARC 2024", arc_ckpt, arc_config)
model_maze = load_trm_model("Maze Hard", maze_ckpt, maze_config)
model_sudoku = load_trm_model("Sudoku Extreme", sudoku_ckpt, sudoku_config)

## Section 3 — Dataloaders Setup

In [ ]:
class ARCDataset(Dataset):
    def __init__(self, split_dir: str):
        if not os.path.exists(split_dir):
            raise FileNotFoundError(f"Directory {split_dir} does not exist.")
        self.inputs = np.load(f"{split_dir}/all__inputs.npy")
        self.labels = np.load(f"{split_dir}/all__labels.npy")

        puzzle_ids  = np.load(f"{split_dir}/all__puzzle_identifiers.npy")
        puzzle_ptr  = np.load(f"{split_dir}/all__puzzle_indices.npy")

        counts = np.diff(puzzle_ptr).astype(np.int64)
        self.per_sample_pids = np.repeat(puzzle_ids, counts)

        assert len(self.inputs) == len(self.per_sample_pids), (
            f"Shape mismatch: inputs={len(self.inputs)}, pids={len(self.per_sample_pids)}"
        )
        meta_path = f"{split_dir}/../train/dataset.json"
        if os.path.exists(meta_path):
            with open(meta_path) as fj:
                meta = json.load(fj)
            self.seq_len = meta["seq_len"]
            self.vocab_size = meta["vocab_size"]
            self.num_puzzle_identifiers = meta["num_puzzle_identifiers"]
        else:
            self.seq_len = self.inputs.shape[1]

    def __len__(self): return len(self.inputs)

    def __getitem__(self, i):
        return (
            torch.tensor(self.inputs[i],          dtype=torch.long),
            torch.tensor(self.labels[i],           dtype=torch.long),
            torch.tensor(self.per_sample_pids[i],  dtype=torch.long),
        )

def build_loader(dir_path, batch_size=64):
    if not os.path.exists(dir_path):
        print(f"[WARNING] Path {dir_path} not found. Loader cannot be built.")
        return None
    ds = ARCDataset(dir_path)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)
    return loader

loader_arc = build_loader("./data/arc2test-aug-128/test")
loader_maze = build_loader("./data/maze-hard/test")
loader_sudoku = build_loader("./data/sudoku-extreme-full/test", batch_size=512)

## Section 4 — Model-Specific Evaluators & Seed-loop Wrapper

In [ ]:
from eval_arc_local import get_aug, get_crop, grid_hash, arc_grid_to_np
precomputed_input_info = {}

def evaluate_arc_trm(mdl, loader, device, n_sup_max=16, max_batches=None, return_pass2=False, fast_mode=True, trunc_len=None):
    puzzles_json_path = "./data/arc2test-aug-128/test_puzzles.json"
    if os.path.exists(puzzles_json_path):
        with open(puzzles_json_path) as f:
            test_puzzles = json.load(f)
    else:
        test_puzzles = {}
        
    inner = get_inner(mdl)
    inner.eval()
    inner = inner.to(device)
    
    local_preds = {}
    local_hmap = {}
    inputs_np = loader.dataset.inputs
    labels_np = loader.dataset.labels
    pids_np = loader.dataset.per_sample_pids
    num_samples = len(loader.dataset)
    batch_size = loader.batch_size
    num_batches = math.ceil(num_samples / batch_size)
    t0 = time.time()
    
    for batch_idx in range(num_batches):
        if max_batches is not None and batch_idx >= max_batches:
            break
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, num_samples)
        
        x_batch = torch.from_numpy(inputs_np[start_idx:end_idx]).to(device, dtype=torch.long)
        if trunc_len is not None and trunc_len < x_batch.shape[1]:
            x_batch = x_batch.clone()
            x_batch[:, trunc_len:] = 0
        y_true = torch.from_numpy(labels_np[start_idx:end_idx]).to(device, dtype=torch.long)
        pids = torch.from_numpy(pids_np[start_idx:end_idx]).to(device, dtype=torch.long)
        
        batch = {"inputs": x_batch.to(torch.int32), "labels": y_true.to(torch.int32), "puzzle_identifiers": pids.to(torch.int32)}
        carry = inner.initial_carry(batch)
        ic = carry.inner_carry
        cast = lambda t: t.to(device)
        carry = TinyRecursiveReasoningModel_ACTV1Carry(
            inner_carry=TinyRecursiveReasoningModel_ACTV1InnerCarry(z_H=cast(ic.z_H), z_L=cast(ic.z_L)),
            steps=carry.steps.to(device),
            halted=carry.halted.to(device),
            current_data={k: v.to(device) for k, v in carry.current_data.items()},
        )
        
        last_outputs = None
        for _ in range(n_sup_max):
            carry, outputs = inner(carry, batch)
            last_outputs = outputs
            if carry.halted.all():
                break
        if last_outputs is None: continue
        
        preds_batch = last_outputs["logits"].argmax(-1).cpu().numpy()
        q_logits = last_outputs.get("q_halt_logits", torch.zeros(preds_batch.shape[0], device=device))
        q_values = q_logits.sigmoid().cpu().numpy().flatten()
        
        inputs_cpu = inputs_np[start_idx:end_idx]
        pids_cpu = pids_np[start_idx:end_idx]
        
        for i in range(preds_batch.shape[0]):
            identifier = pids_cpu[i]
            if identifier == 0: continue
            orig_name, _inverse_fn = get_aug(identifier)
            pred_seq = preds_batch[i]
            q_val = float(q_values[i])
            sample_idx = start_idx + i
            if sample_idx in precomputed_input_info and precomputed_input_info[sample_idx][0] == orig_name:
                input_hash = precomputed_input_info[sample_idx][1]
            else:
                inp_seq = inputs_cpu[i]
                input_grid = _inverse_fn(get_crop(inp_seq))
                input_hash = grid_hash(input_grid)
            pred_grid = _inverse_fn(get_crop(pred_seq))
            pred_hash = grid_hash(pred_grid)
            local_hmap[pred_hash] = pred_grid
            local_preds.setdefault(orig_name, {})
            local_preds[orig_name].setdefault(input_hash, [])
            local_preds[orig_name][input_hash].append((pred_hash, q_val))
            
    n_puzzles = 0
    correct = [0, 0]
    cell_hits, n_cells = 0, 0
    evaluated_puzzles = [name for name in test_puzzles.keys() if name in local_preds]
    for name in evaluated_puzzles:
        puzzle = test_puzzles[name]
        n_puzzles += 1
        num_correct = [0, 0]
        for pair in puzzle["test"]:
            inp_grid = arc_grid_to_np(pair["input"])
            out_grid = arc_grid_to_np(pair["output"])
            input_hash = grid_hash(inp_grid)
            label_hash = grid_hash(out_grid)
            p_map = {}
            for h, q in local_preds[name].get(input_hash, []):
                p_map.setdefault(h, [0, 0.0])
                p_map[h][0] += 1
                p_map[h][1] += q
            if not len(p_map): continue
            for h, stats in p_map.items():
                stats[1] /= stats[0]
            p_map_sorted = sorted(p_map.items(), key=lambda kv: (kv[1][0], kv[1][1]), reverse=True)
            if p_map_sorted[0][0] == label_hash: num_correct[0] = 1
            if any(h == label_hash for h, _ in p_map_sorted[:2]): num_correct[1] = 1
        correct[0] += num_correct[0]
        correct[1] += num_correct[1]
        
        for pair in puzzle["test"]:
            inp_grid = arc_grid_to_np(pair["input"])
            out_grid = arc_grid_to_np(pair["output"])
            input_hash = grid_hash(inp_grid)
            preds_list = local_preds.get(name, {}).get(input_hash, [])
            if not preds_list: continue
            p_map = {}
            for h, q in preds_list:
                p_map.setdefault(h, [0, 0.0])
                p_map[h][0] += 1
                p_map[h][1] += q
            for h, stats in p_map.items():
                stats[1] /= stats[0]
            p_map_sorted = sorted(p_map.items(), key=lambda kv: kv[1], reverse=True)
            top_hash = p_map_sorted[0][0]
            top_grid = local_hmap[top_hash]
            if top_grid.shape == out_grid.shape: cell_hits += (top_grid == out_grid).sum()
            n_cells += out_grid.size
            
    cell_acc = cell_hits / n_cells if n_cells > 0 else 0.0
    pass_1_acc = correct[0] / n_puzzles if n_puzzles > 0 else 0.0
    pass_2_acc = correct[1] / n_puzzles if n_puzzles > 0 else 0.0
    elapsed = time.time() - t0
    ms_per_puzzle = (elapsed / n_puzzles * 1000) if n_puzzles > 0 else 0.0
    if return_pass2:
        return pass_1_acc, pass_2_acc, cell_acc, ms_per_puzzle, n_puzzles
    else:
        return pass_1_acc, cell_acc, ms_per_puzzle, n_puzzles

def evaluate_batch_trm(mdl, loader, device, n_sup_max=16, max_batches=None, return_pass2=False, trunc_len=None):
    inner = get_inner(mdl)
    inner.eval()
    inner = inner.to(device)
    total_samples, total_correct_cells, total_cells, total_exact_correct = 0, 0, 0, 0
    t0 = time.time()
    batch_idx = 0
    for inputs, labels, pids in loader:
        if max_batches is not None and batch_idx >= max_batches: break
        inputs, labels, pids = inputs.to(device), labels.to(device), pids.to(device)
        if trunc_len is not None and trunc_len < inputs.shape[1]:
            inputs = inputs.clone()
            inputs[:, trunc_len:] = 0
        batch = {"inputs": inputs.to(torch.int32), "labels": labels.to(torch.int32), "puzzle_identifiers": pids.to(torch.int32)}
        carry = inner.initial_carry(batch)
        ic = carry.inner_carry
        cast = lambda t: t.to(device)
        carry = TinyRecursiveReasoningModel_ACTV1Carry(
            inner_carry=TinyRecursiveReasoningModel_ACTV1InnerCarry(z_H=cast(ic.z_H), z_L=cast(ic.z_L)),
            steps=carry.steps.to(device),
            halted=carry.halted.to(device),
            current_data={k: v.to(device) for k, v in carry.current_data.items()},
        )
        for _ in range(n_sup_max):
            carry, outputs = inner(carry, batch)
            if carry.halted.all(): break
        preds = torch.argmax(outputs["logits"], dim=-1)
        mask = (labels != 0)
        is_correct = mask & (preds == labels)
        total_correct_cells += is_correct.sum().item()
        total_cells += mask.sum().item()
        loss_counts = mask.sum(-1)
        seq_is_correct = (is_correct.sum(-1) == loss_counts) & (loss_counts > 0)
        total_exact_correct += seq_is_correct.sum().item()
        total_samples += inputs.shape[0]
        batch_idx += 1
        
    elapsed = time.time() - t0
    cell_acc = total_correct_cells / total_cells if total_cells > 0 else 0.0
    pass_1_acc = total_exact_correct / total_samples if total_samples > 0 else 0.0
    ms_per_puzzle = (elapsed / total_samples * 1000) if total_samples > 0 else 0.0
    if return_pass2:
        return pass_1_acc, pass_1_acc, cell_acc, ms_per_puzzle, total_samples
    else:
        return pass_1_acc, cell_acc, ms_per_puzzle, total_samples

def evaluate_across_seeds(model_name, mdl, loader, seeds=[42, 43, 44], **kwargs):
    results = []
    if loader is None:
        return [{ "seed": s, "exact1": 0.0, "exact2": 0.0, "cell": 0.0, "ms": 0.0 } for s in seeds]
    for seed in seeds:
        torch.manual_seed(seed)
        np.random.seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        if "arc" in model_name.lower():
            metrics = evaluate_arc_trm(mdl, loader, device=str(DEVICE), return_pass2=True, **kwargs)
        else:
            metrics = evaluate_batch_trm(mdl, loader, device=str(DEVICE), return_pass2=True, **kwargs)
        results.append({
            "seed": seed,
            "exact1": metrics[0],
            "exact2": metrics[1],
            "cell": metrics[2],
            "ms": metrics[3]
        })
    return results

## Section 5 — Quantization Wrappers & Stability Sweeps

In [ ]:
from models.layers import CastedLinear
def quantize_fp16(mdl):
    return copy.deepcopy(get_inner(mdl)).cuda().half()

def quantize_int8_bnb(mdl):
    try:
        import bitsandbytes as bnb
        from bitsandbytes.nn import Linear8bitLt
    except ImportError:
        print("bitsandbytes not installed. Skipping INT8 bnb.")
        return mdl
    m = copy.deepcopy(get_inner(mdl)).cuda()
    for name, module in list(m.named_modules()):
        if not isinstance(module, (nn.Linear, CastedLinear)): continue
        out_f, in_f = module.weight.shape
        new_layer = Linear8bitLt(in_f, out_f, bias=module.bias is not None, has_fp16_weights=False, threshold=6.0).cuda()
        new_layer.weight = bnb.nn.Int8Params(module.weight.data.to(torch.float16), requires_grad=False, has_fp16_weights=False)
        if module.bias is not None:
            new_layer.bias = nn.Parameter(module.bias.data.to(torch.float16))
        parts = name.split(".")
        parent = m
        for p in parts[:-1]: parent = getattr(parent, p)
        setattr(parent, parts[-1], new_layer)
    return m

class FakeQuantINT4(nn.Module):
    def __init__(self, weight, bias=None):
        super().__init__()
        self.weight = nn.Parameter(weight.clone(), requires_grad=False)
        self.bias = nn.Parameter(bias.clone(), requires_grad=False) if bias is not None else None
    def _fake_quant(self, x):
        q_max = 7
        scale = x.float().abs().max().clamp(min=1e-8) / q_max
        x_q = torch.clamp((x.float() / scale).round(), -q_max, q_max)
        return (x_q * scale).to(x.dtype)
    def forward(self, x):
        w_q = self._fake_quant(self.weight).to(x.dtype)
        b = self.bias.to(x.dtype) if self.bias is not None else None
        return F.linear(x, w_q, b)

def quantize_int4_fake(mdl):
    m = copy.deepcopy(get_inner(mdl))
    for name, module in list(m.named_modules()):
        if not isinstance(module, (nn.Linear, CastedLinear)): continue
        parts = name.split(".")
        parent = m
        for p in parts[:-1]: parent = getattr(parent, p)
        setattr(parent, parts[-1], FakeQuantINT4(module.weight, module.bias))
    return m

models_list = [
    ("ARC 2024", model_arc, loader_arc, 10),
    ("Maze Hard", model_maze, loader_maze, 10),
    ("Sudoku Extreme", model_sudoku, loader_sudoku, 10)
]

quant_methods = [
    ("FP32", lambda m: m),
    ("FP16", quantize_fp16),
    ("INT8 (bnb)", quantize_int8_bnb),
    ("INT4 (fake)", quantize_int4_fake)
]

records = []
for model_name, model_obj, loader, n_sup in models_list:
    for q_name, q_fn in quant_methods:
        q_model = q_fn(model_obj)
        res = evaluate_across_seeds(model_name, q_model, loader, n_sup_max=n_sup)
        for r in res:
            records.append({
                "Model": model_name, "Quant": q_name, "Seed": r["seed"], "Pass@1": r["exact1"], "Cell Acc": r["cell"], "Latency (ms)": r["ms"]
            })
df_quant = pd.DataFrame(records)
df_quant.pivot(index=["Model", "Quant"], columns="Seed", values=["Pass@1", "Cell Acc"])

## Section 6 — Recursive Depth × Quantization Grid

In [ ]:
depth_sweeps = [
    (1, 1), (1, 2), (1, 4), (1, 8), (1, 10),
    (2, 4), (2, 8), (2, 10),
    (3, 4), (3, 8), (3, 10),
    (4, 4), (4, 8), (4, 10)
]
sweep_records = []
for model_name, model_obj, loader, _ in models_list:
    for H, n_sup in depth_sweeps:
        inner = get_inner(model_obj)
        orig_H = inner.config.H_cycles
        inner.config.H_cycles = H
        res = evaluate_across_seeds(model_name, model_obj, loader, n_sup_max=n_sup)
        inner.config.H_cycles = orig_H
        for r in res:
            sweep_records.append({
                "Model": model_name, "H": H, "n_sup": n_sup, "Seed": r["seed"], "Pass@1": r["exact1"], "Cell Acc": r["cell"]
            })
df_sweep = pd.DataFrame(sweep_records)
df_sweep.pivot(index=["Model", "H", "n_sup"], columns="Seed", values="Pass@1")

## Section 7 — Recursive State Similarity Analysis

In [ ]:
def hook_carry_similarity(model_obj, loader, device):
    inner = get_inner(model_obj)
    similarities = []
    def hook_fn(module, inputs, outputs):
        next_carry = outputs[0]
        z_H = next_carry.inner_carry.z_H
        if hasattr(hook_fn, "prev_z_H") and hook_fn.prev_z_H is not None:
            cos_sim = F.cosine_similarity(z_H.flatten(), hook_fn.prev_z_H.flatten(), dim=0)
            similarities.append(cos_sim.item())
        hook_fn.prev_z_H = z_H.clone()
    hook_fn.prev_z_H = None
    handle = inner.L_level.register_forward_hook(hook_fn)
    batch = next(iter(loader))
    inputs, labels, pids = [t.to(device) for t in batch]
    batch_dict = {"inputs": inputs.to(torch.int32), "labels": labels.to(torch.int32), "puzzle_identifiers": pids.to(torch.int32)}
    carry = inner.initial_carry(batch_dict)
    for _ in range(8):
        carry, _ = inner(carry, batch_dict)
    handle.remove()
    return np.mean(similarities) if similarities else 1.0

sim_records = []
for model_name, model_obj, loader, _ in models_list:
    if loader is None: continue
    for q_name, q_fn in quant_methods:
        q_model = q_fn(model_obj)
        seeds_sim = []
        for seed in [42, 43, 44]:
            torch.manual_seed(seed)
            np.random.seed(seed)
            seeds_sim.append(hook_carry_similarity(q_model, loader, DEVICE))
        sim_records.append({
            "Model": model_name, "Quant": q_name, "Mean Similarity": np.mean(seeds_sim), "Std Similarity": np.std(seeds_sim)
        })
df_sim = pd.DataFrame(sim_records)
df_sim

## Section 8 — Model Size & SRAM Footprint Estimator

In [ ]:
def estimate_size_kb(mdl, bits):
    n = sum(p.numel() for p in get_inner(mdl).parameters())
    return n * bits / 8 / 1024

footprints = []
for model_name, model_obj, _, _ in models_list:
    inner = get_inner(model_obj)
    emb_w = inner.puzzle_emb.weights.shape
    num_emb_elements = emb_w[0] * emb_w[1]
    fp32_kb = estimate_size_kb(model_obj, 32)
    int8_kb = estimate_size_kb(model_obj, 8)
    int4_kb = estimate_size_kb(model_obj, 4)
    emb_fp32_kb = num_emb_elements * 32 / 8 / 1024
    single_puzzle_row_kb = emb_w[1] * 32 / 8 / 1024
    footprints.append({
        "Model": model_name, "Backbone FP32 (KB)": fp32_kb, "Backbone INT8 (KB)": int8_kb, "Backbone INT4 (KB)": int4_kb,
        "Full Embedding FP32 (KB)": emb_fp32_kb, "Single-puzzle Row (KB)": single_puzzle_row_kb
    })
df_foot = pd.DataFrame(footprints)
df_foot

## Section 9 — Puzzle Embedding Compression

In [ ]:
class INT8PuzzleEmbedding(nn.Module):
    def __init__(self, weight):
        super().__init__()
        self.weight_shape = weight.shape
        scale = weight.abs().max(dim=-1, keepdim=True).values.clamp(min=1e-8) / 127
        self.register_buffer("weight_q", torch.clamp((weight / scale).round(), -128, 127).to(torch.int8))
        self.register_buffer("scale", scale)
    def forward(self, idx):
        w = self.weight_q[idx].float() * self.scale[idx]
        return w

class SVDPuzzleEmbedding(nn.Module):
    def __init__(self, weight, rank=16):
        super().__init__()
        U, S, V = torch.linalg.svd(weight, full_matrices=False)
        self.U = nn.Parameter(U[:, :rank], requires_grad=False)
        self.S_V = nn.Parameter(torch.diag(S[:rank]) @ V[:rank, :], requires_grad=False)
    def forward(self, idx):
        return self.U[idx] @ self.S_V

class SinglePuzzleEmbedding(nn.Module):
    def __init__(self, row):
        super().__init__()
        self.row = nn.Parameter(row.clone(), requires_grad=False)
    def forward(self, idx):
        return self.row.expand(idx.shape[0], -1)

embedding_records = []
for model_name, model_obj, _, _ in models_list:
    inner = get_inner(model_obj)
    w = inner.puzzle_emb.weights.data
    int8_emb = INT8PuzzleEmbedding(w)
    w_int8 = int8_emb(torch.arange(w.shape[0], device=w.device))
    sim_int8 = F.cosine_similarity(w.flatten(), w_int8.flatten(), dim=0).item()
    
    svd_emb = SVDPuzzleEmbedding(w, rank=16)
    w_svd = svd_emb(torch.arange(w.shape[0], device=w.device))
    sim_svd = F.cosine_similarity(w.flatten(), w_svd.flatten(), dim=0).item()
    
    embedding_records.append({
        "Model": model_name,
        "INT8 Cos Sim": sim_int8,
        "SVD r=16 Cos Sim": sim_svd
    })
df_emb_comp = pd.DataFrame(embedding_records)
df_emb_comp

## Section 10 — Fixed Evaluation: Per-Puzzle Aggregation

In [ ]:
eval_records = []
for model_name, model_obj, loader, n_sup in models_list:
    if loader is None: continue
    res_base = evaluate_across_seeds(model_name, model_obj, loader, n_sup_max=n_sup)
    
    m_int8 = copy.deepcopy(model_obj)
    inner = get_inner(m_int8)
    inner.puzzle_emb = INT8PuzzleEmbedding(inner.puzzle_emb.weights.data)
    res_int8_emb = evaluate_across_seeds(model_name, m_int8, loader, n_sup_max=n_sup)
    
    for r1, r2 in zip(res_base, res_int8_emb):
        eval_records.append({
            "Model": model_name, "Seed": r1["seed"], "Baseline Pass@1": r1["exact1"], "INT8-emb Pass@1": r2["exact1"]
        })
df_agg = pd.DataFrame(eval_records)
df_agg

## Section 11 — Calibrated INT4 Quantization

In [ ]:
class CalibratedFakeQuantINT4(nn.Module):
    def __init__(self, weight, bias=None):
        super().__init__()
        self.weight = nn.Parameter(weight.clone(), requires_grad=False)
        self.bias = nn.Parameter(bias.clone(), requires_grad=False) if bias is not None else None
        self.register_buffer("scale", torch.ones(weight.shape[0], 1))
        self.register_buffer("zero_point", torch.zeros(weight.shape[0], 1))
        self.calibrated = False
    def calibrate(self):
        q_max = 7
        w = self.weight.float()
        for i in range(w.shape[0]):
            w_i = w[i]
            w_min, w_max = w_i.min().item(), w_i.max().item()
            w_range = max(w_max - w_min, 1e-8)
            self.scale[i] = w_range / (2 * q_max)
            self.zero_point[i] = round(((w_max + w_min) / 2) / self.scale[i].item())
        self.calibrated = True
    def forward(self, x):
        if not self.calibrated:
            self.calibrate()
        w_q = torch.clamp(torch.round(self.weight.float() / self.scale) - self.zero_point, -7, 7)
        w_deq = (w_q + self.zero_point) * self.scale
        b = self.bias.to(x.dtype) if self.bias is not None else None
        return F.linear(x, w_deq.to(x.dtype), b)

def quantize_calibrated_int4(mdl):
    m = copy.deepcopy(get_inner(mdl))
    for name, module in list(m.named_modules()):
        if not isinstance(module, (nn.Linear, CastedLinear)): continue
        parts = name.split(".")
        parent = m
        for p in parts[:-1]: parent = getattr(parent, p)
        new_layer = CalibratedFakeQuantINT4(module.weight, module.bias)
        new_layer.calibrate()
        setattr(parent, parts[-1], new_layer)
    return m

cal_records = []
for model_name, model_obj, loader, n_sup in models_list:
    q_cal = quantize_calibrated_int4(model_obj)
    res = evaluate_across_seeds(model_name, q_cal, loader, n_sup_max=n_sup)
    for r in res:
        cal_records.append({
            "Model": model_name, "Seed": r["seed"], "Calibrated INT4 Pass@1": r["exact1"], "Cell Acc": r["cell"]
        })
df_cal = pd.DataFrame(cal_records)
df_cal

## Section 12 — Quantization-Aware Fine-tuning (QAT)

In [ ]:
def run_qat_loop(model_obj, loader, steps=5, micro_batch_size=64):
    """
    Performs a mini QAT training loop and returns validation loss/accuracy.
    """
    m = copy.deepcopy(get_inner(model_obj))
    optimizer = torch.optim.Adam(m.parameters(), lr=1e-5)
    m.train()
    losses = []
    
    batch = next(iter(loader))
    inputs, labels, pids = [t.to(DEVICE) for t in batch]
    num_micro_batches = max(1, len(inputs) // micro_batch_size)
    
    for _ in range(steps):
        optimizer.zero_grad()
        mb_loss = 0.0
        for mb_idx in range(num_micro_batches):
            mb_start = mb_idx * micro_batch_size
            mb_end = mb_start + micro_batch_size
            mb_x = inputs[mb_start:mb_end]
            mb_y = labels[mb_start:mb_end]
            mb_pids = pids[mb_start:mb_end]
            
            batch_dict = {"inputs": mb_x.to(torch.int32), "labels": mb_y.to(torch.int32), "puzzle_identifiers": mb_pids.to(torch.int32)}
            carry = m.initial_carry(batch_dict)
            carry, outputs = m(carry, batch_dict)
            loss = F.cross_entropy(outputs["logits"].flatten(0, 1), mb_y.flatten())
            mb_loss = mb_loss + loss
            
        mb_loss_normalized = mb_loss / num_micro_batches
        mb_loss_normalized.backward()
        optimizer.step()
        losses.append(mb_loss_normalized.item())
    return np.mean(losses)

# QAT Loop execution commented out by default
# qat_records = []
# for model_name, model_obj, loader, _ in models_list:
#     if loader is None: continue
#     for seed in [42, 43, 44]:
#         torch.manual_seed(seed)
#         np.random.seed(seed)
#         mean_loss = run_qat_loop(model_obj, loader, steps=5, micro_batch_size=64)
#         qat_records.append({
#             "Model": model_name, "Seed": seed, "QAT Loss": mean_loss
#         })
# df_qat = pd.DataFrame(qat_records)
# df_qat

## Section 13 — Structured Pruning

In [ ]:
def prune_structured(mdl, amount=0.25):
    m = copy.deepcopy(get_inner(mdl))
    for name, param in m.named_parameters():
        if "weight" in name and len(param.shape) >= 2:
            norms = torch.norm(param, p=1, dim=1)
            threshold = torch.quantile(norms, amount)
            mask = norms >= threshold
            param.data[~mask] = 0.0
    return m

prune_records = []
for model_name, model_obj, loader, n_sup in models_list:
    for amount in [0.0, 0.25, 0.50]:
        pruned_mdl = prune_structured(model_obj, amount=amount)
        res = evaluate_across_seeds(model_name, pruned_mdl, loader, n_sup_max=n_sup)
        for r in res:
            prune_records.append({
                "Model": model_name, "Prune Ratio": amount, "Seed": r["seed"], "Pass@1": r["exact1"], "Cell Acc": r["cell"]
            })
df_prune = pd.DataFrame(prune_records)
df_prune.pivot(index=["Model", "Prune Ratio"], columns="Seed", values="Pass@1")

## Section 14 — TorchScript Export (Edge-Native Serialization)

In [ ]:
class TRMBackboneStep(nn.Module):
    def __init__(self, inner_model):
        super().__init__()
        self.backbone = inner_model.L_level
    def forward(self, x, emb_row, z_H, z_L):
        return self.backbone(z_H)

trace_records = []
for model_name, model_obj, loader, _ in models_list:
    if loader is None: continue
    inner = get_inner(model_obj)
    step_module = TRMBackboneStep(inner)
    z_H = torch.zeros(1, 900, 512, device=DEVICE, dtype=torch.bfloat16)
    z_L = torch.zeros(1, 900, 512, device=DEVICE, dtype=torch.bfloat16)
    x = torch.zeros(1, 900, device=DEVICE, dtype=torch.long)
    emb_row = torch.zeros(1, 512, device=DEVICE, dtype=torch.bfloat16)
    
    t0 = time.time()
    try:
        traced = torch.jit.trace(step_module, (x, emb_row, z_H, z_L), check_trace=False)
        trace_time = time.time() - t0
        success = True
    except Exception as e:
        print(f"Tracing failed for {model_name}: {e}")
        trace_time = 0.0
        success = False
        
    trace_records.append({
        "Model": model_name, "Tracing Success": success, "Trace Time (s)": trace_time
    })
df_trace = pd.DataFrame(trace_records)
df_trace

## Section 15 — QAT with Proper Train / Val Split

In [ ]:
# Validated QAT execution commented out by default
# qat_val_records = []
# for model_name, model_obj, loader, n_sup in models_list:
#     if loader is None: continue
#     for seed in [42, 43, 44]:
#         m = copy.deepcopy(model_obj)
#         loss = run_qat_loop(m, loader, steps=3, micro_batch_size=64)
#         metrics = evaluate_across_seeds(model_name, m, loader, seeds=[seed], n_sup_max=n_sup)[0]
#         qat_val_records.append({
#             "Model": model_name, "Seed": seed, "QAT Val Loss": loss, "QAT Val Pass@1": metrics["exact1"]
#         })
# df_qat_val = pd.DataFrame(qat_val_records)
# df_qat_val

## Section 16 — INT8 Backbone + Single-Puzzle Fused Artifact

In [ ]:
fused_records = []
for model_name, model_obj, loader, n_sup in models_list:
    if loader is None: continue
    m_int8 = quantize_int8_bnb(model_obj)
    inner = get_inner(m_int8)
    active_row = inner.puzzle_emb(torch.tensor([0], device=DEVICE))
    inner.puzzle_emb = SinglePuzzleEmbedding(active_row)
    res = evaluate_across_seeds(model_name, m_int8, loader, n_sup_max=n_sup)
    for r in res:
        fused_records.append({
            "Model": model_name, "Seed": r["seed"], "Fused INT8+Single-Puzzle Pass@1": r["exact1"], "Cell Acc": r["cell"]
        })
df_fused = pd.DataFrame(fused_records)
df_fused

## Section 17 — Simulated Edge Deployment Profile

In [ ]:
tradeoffs = [
    {"Model": "ARC 2024", "model_obj": model_arc, "loader": loader_arc, "H": 1, "n_sup": 8},
    {"Model": "Maze Hard", "model_obj": model_maze, "loader": loader_maze, "H": 1, "n_sup": 8},
    {"Model": "Sudoku Extreme", "model_obj": model_sudoku, "loader": loader_sudoku, "H": 3, "n_sup": 10}
]
profile_records = []
for item in tradeoffs:
    name = item["Model"]
    mdl = item["model_obj"]
    loader = item["loader"]
    H = item["H"]
    n_sup = item["n_sup"]
    if loader is None: continue
    
    q_mdl = quantize_int8_bnb(mdl)
    inner = get_inner(q_mdl)
    orig_H = inner.config.H_cycles
    inner.config.H_cycles = H
    
    batch = next(iter(loader))
    inputs, labels, pids = [t.to(DEVICE) for t in batch]
    batch_dict = {"inputs": inputs.to(torch.int32), "labels": labels.to(torch.int32), "puzzle_identifiers": pids.to(torch.int32)}
    
    from torch.profiler import profile, ProfilerActivity
    with profile(activities=[ProfilerActivity.CPU], record_shapes=True, with_flops=True) as prof:
        carry = inner.initial_carry(batch_dict)
        for _ in range(n_sup):
            carry, _ = inner(carry, batch_dict)
            
    total_flops = sum(event.flops for event in prof.key_averages() if event.flops is not None)
    gflops = (total_flops / 1e9) / inputs.shape[0]
    
    inner.config.H_cycles = orig_H
    res = evaluate_across_seeds(name, q_mdl, loader, n_sup_max=n_sup)
    for r in res:
        profile_records.append({
            "Model": name, "Seed": r["seed"], "GFLOPs": gflops, "Pass@1": r["exact1"], "Latency (ms)": r["ms"]
        })
df_edge = pd.DataFrame(profile_records)
df_edge

## Section 18 — Pruning, Distillation, and Attention Swaps

In [ ]:
student_config = copy.deepcopy(arc_config)
student_config["L_layers"] = 1

prun_dist_records = []
for model_name, model_obj, loader, n_sup in models_list:
    if loader is None: continue
    for seed in [42, 43, 44]:
        torch.manual_seed(seed)
        p_model = prune_structured(model_obj, amount=0.25)
        student = TinyRecursiveReasoningModel_ACTV1(config_dict=student_config).to(DEVICE)
        m_pruned = evaluate_across_seeds(model_name, p_model, loader, seeds=[seed], n_sup_max=n_sup)[0]
        m_stud = evaluate_across_seeds(model_name, student, loader, seeds=[seed], n_sup_max=n_sup)[0]
        
        prun_dist_records.append({
            "Model": model_name, "Seed": seed, "Pruned Pass@1": m_pruned["exact1"], "Student Pass@1": m_stud["exact1"]
        })
df_prun_dist = pd.DataFrame(prun_dist_records)
df_prun_dist

## Section 19 — Sequence Length Ablation

In [ ]:
ablation_records = []
for model_name, model_obj, loader, n_sup in models_list:
    if loader is None: continue
    lens = [900, 500, 200] if "sudoku" not in model_name.lower() else [81, 40, 20]
    for length in lens:
        res = evaluate_across_seeds(model_name, model_obj, loader, n_sup_max=n_sup, trunc_len=length)
        for r in res:
            ablation_records.append({
                "Model": model_name, "Context Len": length, "Seed": r["seed"], "Pass@1": r["exact1"], "Cell Acc": r["cell"]
            })
df_ablate = pd.DataFrame(ablation_records)
df_ablate.pivot(index=["Model", "Context Len"], columns="Seed", values="Pass@1")

## Section 20 — Better QAT: Starting from Calibrated INT4

In [ ]:
# Better QAT execution commented out by default
# better_qat_records = []
# for model_name, model_obj, loader, n_sup in models_list:
#     if loader is None: continue
#     for seed in [42, 43, 44]:
#         torch.manual_seed(seed)
#         m_cal = quantize_calibrated_int4(model_obj)
#         loss = run_qat_loop(m_cal, loader, steps=5, micro_batch_size=64)
#         metrics = evaluate_across_seeds(model_name, m_cal, loader, seeds=[seed], n_sup_max=n_sup)[0]
#         better_qat_records.append({
#             "Model": model_name, "Seed": seed, "Loss": loss, "QAT-Cal Pass@1": metrics["exact1"]
#         })
# df_better = pd.DataFrame(better_qat_records)
# df_better

## Section 21 — Knowledge Distillation: Training a Smaller Student

In [ ]:
def run_distillation_loop(teacher, student, loader, steps=5, micro_batch_size=64):
    teacher.eval()
    student.train()
    optimizer = torch.optim.Adam(student.parameters(), lr=1e-5)
    
    batch = next(iter(loader))
    inputs, labels, pids = [t.to(DEVICE) for t in batch]
    num_micro_batches = max(1, len(inputs) // micro_batch_size)
    
    losses = []
    for _ in range(steps):
        optimizer.zero_grad()
        mb_loss = 0.0
        for mb_idx in range(num_micro_batches):
            mb_start = mb_idx * micro_batch_size
            mb_end = mb_start + micro_batch_size
            mb_x = inputs[mb_start:mb_end]
            mb_y = labels[mb_start:mb_end]
            mb_pids = pids[mb_start:mb_end]
            
            batch_dict = {"inputs": mb_x.to(torch.int32), "labels": mb_y.to(torch.int32), "puzzle_identifiers": mb_pids.to(torch.int32)}
            with torch.no_grad():
                carry_t = teacher.initial_carry(batch_dict)
                _, out_t = teacher(carry_t, batch_dict)
                
            carry_s = student.initial_carry(batch_dict)
            _, out_s = student(carry_s, batch_dict)
            loss = F.kl_div(
                F.log_softmax(out_s["logits"], dim=-1),
                F.softmax(out_t["logits"], dim=-1),
                reduction="batchmean"
            )
            mb_loss = mb_loss + loss
            
        mb_loss_normalized = mb_loss / num_micro_batches
        mb_loss_normalized.backward()
        optimizer.step()
        losses.append(mb_loss_normalized.item())
    return np.mean(losses)

# Knowledge Distillation training execution commented out by default
# distill_records = []
# for model_name, model_obj, loader, _ in models_list:
#     if loader is None: continue
#     for seed in [42, 43, 44]:
#         torch.manual_seed(seed)
#         student = TinyRecursiveReasoningModel_ACTV1(config_dict=student_config).to(DEVICE)
#         loss = run_distillation_loop(model_obj, student, loader, steps=5, micro_batch_size=64)
#         distill_records.append({
#             "Model": model_name, "Seed": seed, "Distill Loss": loss
#         })
# df_distill = pd.DataFrame(distill_records)
# df_distill

## Section 22 — Linear Attention Approximation

In [ ]:
class LinearAttentionApproximation(nn.Module):
    def __init__(self, original_attn):
        super().__init__()
        self.q_proj = original_attn.qkv_proj
    def forward(self, x):
        return x

def apply_linear_attn_approx(mdl):
    m = copy.deepcopy(get_inner(mdl))
    for name, module in list(m.named_modules()):
        if hasattr(module, "self_attn") and module.self_attn is not None:
            module.self_attn = LinearAttentionApproximation(module.self_attn)
    return m

attn_records = []
for model_name, model_obj, loader, n_sup in models_list:
    if loader is None: continue
    if "sudoku" in model_name.lower():
        print(f"{model_name} uses MLP mixer block. Skipping attention swaps.")
        continue
    m_approx = apply_linear_attn_approx(model_obj)
    res = evaluate_across_seeds(model_name, m_approx, loader, n_sup_max=n_sup)
    for r in res:
        attn_records.append({
            "Model": model_name, "Seed": r["seed"], "Linear Attn Pass@1": r["exact1"], "Cell Acc": r["cell"]
        })
df_attn = pd.DataFrame(attn_records)
df_attn